<a href="https://colab.research.google.com/github/safakatakancelik/portfolio-public/blob/main/notebooks/building_attention_from_scratch/attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementing Attention

1. Raw attention with word embeddings, no Q/K/V
2. Add Q, K, V matrices
3. Scaled dot-product (divide by √d_k, show softmax saturation without it)
4. Causal masking (set future positions to −∞ before softmax)
5. Multi-head attention (split dims across heads)
6. Training (parallel, masked) vs inference (sequential, KV cache)
7. Inspect attention matrices on ambiguous sentences
8. Rewrite as PyTorch nn.Module inside a transformer block (attention → residual → LayerNorm → FFN)

### Learning Goals
- understand raw attention vs learned Q/K/V projections
- see why √d_k scaling prevents softmax saturation
- understand causal masking for autoregressive generation
- use MHA as context for showing KV cache benefits
- compare training-time parallelism vs inference-time sequential decoding
- read attention maps to see what the model "looks at"
- bridge from numpy prototype to production-style PyTorch code


$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


---

sources used:
https://machinelearningmastery.com/the-attention-mechanism-from-scratch/

In [ ]:
!pip install torch
!pip install torchtext

In [1]:
import numpy as np


from scipy.special import softmax

### Step 1: Raw Attention

In [20]:
np.random.seed(42)

english_words = ["the", "cat", "sat", "on", "the", "mat"]

# building the vocabulary with random vectors
dim = 8
unique_words = dict.fromkeys(english_words)
vocab = {w: np.random.randn(dim) for w in unique_words}


X = np.array([vocab[w] for w in english_words])  # (6, 8)
d_k = X.shape[1]

scores = X @ X.T / np.sqrt(d_k)  # (6, 8) @ (8, 6) -> (6, 6) attention score for each word pair

print("Tokens:", english_words)
print("\nScores shape:", scores.shape)
print("\nRaw attention scores:")
print(np.round(scores, 2))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)

Raw attention scores:
[[ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-1.44  2.81  1.13  0.38 -1.44  1.98]
 [-1.61  1.13  2.89 -0.85 -1.61  0.37]
 [ 0.08  0.38 -0.85  2.13  0.08  0.03]
 [ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-0.96  1.98  0.37  0.03 -0.96  3.17]]


---


## Step 2: Add Q, K, V matrices

In [21]:
# np.random.randn(4)

array([ 0.73846658,  0.17136828, -0.11564828, -0.3011037 ])